# SQL to Pandas

A short hands-on walkthrough for analysts who already know SQL.

**Main idea:** use SQL to extract a useful analysis-ready dataset, then use Pandas to answer many follow-up questions without repeatedly changing and rerunning the SQL.

## 1. Setup

Add the Teradata connection code above or below this cell as needed.

For example, once a connection object exists, the notebook will use:

```python
df = pd.read_sql(query, connection)
```

For a quick copy/paste workflow from Toad, you can also use:

```python
df = pd.read_clipboard()
```

In [ ]:
import pandas as pd

# Add your Teradata connection here.
# Example:
# connection = ...


## 2. Build the SQL extract

The goal is to bring back an analysis-ready inpatient / observation authorization dataset.

Do the heavy filtering in Teradata first:
- Date range
- Bed types
- Data source
- Managed population
- Market
- Managing entity

Then use Pandas for rapid exploration and follow-up analysis.

In [ ]:
query = """
SELECT
          IP.BedType
        , IP.DataSource
        , IP.OperationalMarket
        , IP.OperationalSubMarket
        , IP.ReportingPod
        , IP.ManagingEntity
        , IP.ManagingProviderName
        , IP.PodCd
        , IP.PodName
        , IP.PCPName
        , IP.PCPNPI
        , IP.AuthznKey
        , OREPLACE(IP.AuthznKey, '5|', '') AS AuthznKey_Clean
        , IP.MemberID
        , YEAR(IP.Admit) AS AdmitYear
        , IP.Admit
        , IP.Discharge
        , IP.Admit - EXTRACT(DAY FROM IP.Admit) + 1 AS AdmitMonth
        , CASE
              WHEN IP.Discharge IS NOT NULL
              THEN IP.Discharge - EXTRACT(DAY FROM IP.Discharge) + 1
              ELSE NULL
          END AS DischargeMonth
        , IP.Discharge - IP.Admit AS LengthOfStay
        , IP.AdmittingProviderName AS AdmittingFacilityName
        , IP.AdmittingProviderNPI  AS AdmittingFacilityNPI
        , IP.AdmissionReason
        , IP.Expired
        , IP.MemberDOB
        , IP.HCODE
        , IP.PlanType
        , IP.AdmitDischargeError
        , IP.PrimaryDiagnosisCode
        , IP.PrimaryDiagnosis
        , IP.LACE
        , IP.AdmissionType
        , IP.AdmitFrom
        , IP.DischargeStatusInd
        , IP.DischargeStatusCode
        , IP.DischargeStatusDescription
        , CASE
              WHEN IP.DischargeStatusCode IN ('03', '61', '62', '63') THEN 'SNF/Rehab'
              WHEN IP.DischargeStatusCode IN ('06', '50', '51')       THEN 'Home Health'
              WHEN IP.DischargeStatusCode = '01'                      THEN 'Home'
              ELSE 'Other'
          END AS Discharge_Disposition_Group
FROM BISDM_CA_BASE_PRD.MSO_Core_Utilization_Detail AS IP
CROSS JOIN vt_params AS P
WHERE IP.Admit BETWEEN P.StartDate AND P.EndDate
  AND IP.BedType IN ('Acute', 'OBS', 'LTAC')
  AND IP.DataSource = 'Authorizations'
  AND IP.CareAlliesManagedFlag = 1
  AND IP.OperationalMarket = 'TX'
  AND IP.ManagingEntity = 'VALLEY ORGANIZED PHYSICIANS LLC (VOP)'
"""


### Pull the data

Use either approach below.

**Preferred:** read directly from Teradata.

```python
df = pd.read_sql(query, connection)
```

**Quick training shortcut:** run the SQL in Toad, copy the result grid, then:

```python
df = pd.read_clipboard()
```

`read_clipboard()` is especially useful when you want to start analyzing immediately without changing your normal SQL workflow.

In [ ]:
# Preferred once your connection is configured:
# df = pd.read_sql(query, connection)

# Quick Toad -> Pandas option:
# df = pd.read_clipboard()


## 3. First look at the DataFrame

Think of a Pandas **DataFrame** as the result grid returned by SQL.

These few commands answer the first questions you usually have about a new extract.

In [ ]:
# First few rows
df.head()


In [ ]:
# (rows, columns)
df.shape


In [ ]:
# Column names
df.columns.tolist()


In [ ]:
# Data types, null counts, and memory overview
df.info()


## 4. Fast counts without rewriting SQL

This is one of the biggest productivity gains for SQL analysts.

Instead of changing the query to `COUNT(*)`, `GROUP BY`, rerunning it, and waiting for another result set, analyze the DataFrame already in memory.

In [ ]:
# Total rows
len(df)


In [ ]:
# Same idea, plus number of columns
df.shape


### `value_counts()`

SQL equivalent:

```sql
SELECT BedType, COUNT(*)
FROM ...
GROUP BY BedType
ORDER BY COUNT(*) DESC;
```

Pandas: one line.

In [ ]:
df["BedType"].value_counts()


In [ ]:
# Breakdown by submarket
df["OperationalSubMarket"].value_counts()


In [ ]:
# Breakdown by reporting pod
df["ReportingPod"].value_counts().head(10)


In [ ]:
# Most common facilities
df["AdmittingFacilityName"].value_counts().head(10)


In [ ]:
# Discharge disposition mix
df["Discharge_Disposition_Group"].value_counts()


### Percentages instead of counts

`normalize=True` turns counts into proportions.

Multiply by 100 for percentages.

In [ ]:
df["BedType"].value_counts(normalize=True) * 100


## 5. Distinct counts with `nunique()`

SQL:

```sql
SELECT COUNT(DISTINCT MemberID)
FROM ...
```

Pandas:

In [ ]:
df["MemberID"].nunique()


In [ ]:
# Number of PCPs
df["PCPNPI"].nunique()


In [ ]:
# Number of admitting facilities
df["AdmittingFacilityNPI"].nunique()


## 6. Filtering. Think `WHERE`

SQL:

```sql
WHERE BedType = 'OBS'
```

Pandas:

In [ ]:
obs = df[df["BedType"] == "OBS"]

obs.head()


Once you create a filtered DataFrame, you can keep asking questions about it without going back to Teradata.

In [ ]:
# OBS by submarket
obs["OperationalSubMarket"].value_counts()


In [ ]:
# OBS by reporting pod
obs["ReportingPod"].value_counts().head(10)


In [ ]:
# Top OBS admitting facilities
obs["AdmittingFacilityName"].value_counts().head(10)


### Multiple conditions

SQL:

```sql
WHERE BedType = 'Acute'
  AND AdmitYear = 2026
```

Pandas uses `&` for AND.

In [ ]:
acute_2026 = df[
    (df["BedType"] == "Acute")
    & (df["AdmitYear"] == 2026)
]

acute_2026.head()


## 7. `describe()`. Instant numerical profiling

Instead of writing SQL for `COUNT`, `AVG`, `MIN`, `MAX`, standard deviation, and percentiles, use one command.

In [ ]:
df["LengthOfStay"].describe()


For multiple numerical columns:

In [ ]:
df[
    [
        "LengthOfStay",
        "LACE"
    ]
].describe()


### Grouped `describe()`

This is particularly useful for exploratory analysis.

Question: **How does length of stay differ across Acute, OBS, and LTAC?**

In [ ]:
df.groupby("BedType")["LengthOfStay"].describe()


One line gives you:
- count
- mean
- standard deviation
- minimum
- 25th percentile
- median
- 75th percentile
- maximum

Doing the equivalent in SQL usually takes much more code.

## 8. Simple `GROUP BY`

Start with the Pandas equivalent of:

```sql
SELECT BedType, COUNT(*)
FROM ...
GROUP BY BedType;
```

In [ ]:
df.groupby("BedType").size()


Two grouping fields:

In [ ]:
df.groupby(
    ["OperationalSubMarket", "BedType"]
).size()


Turn the result back into a clean DataFrame:

In [ ]:
bedtype_summary = (
    df
    .groupby(["OperationalSubMarket", "BedType"])
    .size()
    .reset_index(name="Admissions")
)

bedtype_summary.head(10)


## 9. Multiple aggregations with `.agg()`

This is the closest equivalent to a typical SQL analytical `GROUP BY`.

Question: **What does utilization look like by submarket and bed type?**

In [ ]:
summary = (
    df
    .groupby(["OperationalSubMarket", "BedType"])
    .agg(
        Admissions=("AuthznKey_Clean", "nunique"),
        Members=("MemberID", "nunique"),
        PCPs=("PCPNPI", "nunique"),
        Avg_LengthOfStay=("LengthOfStay", "mean"),
        Avg_LACE=("LACE", "mean")
    )
    .reset_index()
)

summary


### Add a sort

SQL:

```sql
ORDER BY Admissions DESC
```

Pandas:

In [ ]:
summary.sort_values(
    "Admissions",
    ascending=False
).head(10)


## 10. `crosstab()`. Fast two-way analysis

This is a very useful shortcut for analysts.

Question: **How do bed types vary across submarkets?**

In [ ]:
pd.crosstab(
    df["OperationalSubMarket"],
    df["BedType"]
)


Now show row percentages instead of counts:

In [ ]:
pd.crosstab(
    df["OperationalSubMarket"],
    df["BedType"],
    normalize="index"
) * 100


Another useful example:

**What is the discharge disposition mix by bed type?**

In [ ]:
pd.crosstab(
    df["BedType"],
    df["Discharge_Disposition_Group"]
)


## 11. Dates are easier once the extract is in Pandas

Make sure date fields are recognized as dates.

In [ ]:
date_cols = [
    "Admit",
    "Discharge",
    "AdmitMonth",
    "DischargeMonth",
    "MemberDOB"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")


Now answering a monthly utilization question does not require changing the SQL query.

In [ ]:
monthly = (
    df
    .groupby(["AdmitMonth", "BedType"])
    .agg(
        Admissions=("AuthznKey_Clean", "nunique"),
        Members=("MemberID", "nunique")
    )
    .reset_index()
)

monthly.head(12)


## 12. A practical exploration workflow

Once the SQL extract is loaded, a good first pass is often just:

```python
df.shape
df.head()
df.info()

df["BedType"].value_counts()
df["OperationalSubMarket"].value_counts()
df["AdmittingFacilityName"].value_counts().head(10)

df["MemberID"].nunique()
df["PCPNPI"].nunique()

df["LengthOfStay"].describe()
df.groupby("BedType")["LengthOfStay"].describe()
```

That can answer many initial business questions before you write any additional SQL.

## 13. Practice exercise

Using only the existing `df`, answer the following:

1. How many authorization events are in the extract?
2. How many unique members?
3. What percentage of events are Acute, OBS, and LTAC?
4. Which 10 admitting facilities have the most authorization events?
5. Which reporting pods have the most OBS events?
6. Compare average length of stay across bed types.
7. What is the discharge disposition mix for Acute vs OBS?
8. Build a summary by `OperationalSubMarket` and `BedType` containing:
   - unique authorizations
   - unique members
   - unique PCPs
   - average length of stay
   - average LACE
9. Sort the result from highest to lowest number of authorizations.


## 14. Key takeaway

### SQL
Use SQL to:
- restrict the population
- join large source tables
- filter dates
- select the fields you need
- perform database-scale processing

### Pandas
Use Pandas to:
- inspect
- count
- profile
- filter
- compare
- aggregate
- test follow-up questions
- prepare results for visualization, statistics, automation, or machine learning

**Extract once. Explore many times.**
